# 04 Retention and Change Tracking

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
CATALOG_SCHEMA="governance"; RETENTION_DAYS=90

dim_table=spark.table(f"{CATALOG_SCHEMA}.dim_data_object")
dim_col=spark.table(f"{CATALOG_SCHEMA}.dim_column")

snapshot_table=(dim_table.groupBy('layer','domain','workspace_name','lakehouse_name','schema_name')
 .agg(F.countDistinct('data_object_id').alias('total_tables'),
      F.sum(F.when(F.col('table_definition_status')=='Missing',1).otherwise(0)).alias('missing_table_description_count'),
      F.sum(F.when(F.col('grain_definition_status')=='Missing',1).otherwise(0)).alias('missing_grain_count')))
snapshot_col=(dim_col.groupBy('layer','workspace_name','lakehouse_name','schema_name')
 .agg(F.count('*').alias('total_columns'),
      F.sum(F.when(F.col('column_definition_status')=='Missing',1).otherwise(0)).alias('missing_column_definition_count')))
snapshot=(snapshot_table.join(snapshot_col,on=['layer','workspace_name','lakehouse_name','schema_name'],how='left')
 .withColumn('snapshot_at',F.current_timestamp())
 .withColumn('column_definition_coverage_pct',F.when(F.col('total_columns')>0,(F.col('total_columns')-F.col('missing_column_definition_count'))/F.col('total_columns')).otherwise(F.lit(None).cast('double'))))
snapshot.write.format('delta').mode('append').saveAsTable(f'{CATALOG_SCHEMA}.fact_definition_quality_snapshot')

for t in [f'{CATALOG_SCHEMA}.stg_table_metadata',f'{CATALOG_SCHEMA}.stg_column_metadata']:
    df=spark.table(t)
    if 'scanned_at' in df.columns:
        df.where(F.col('scanned_at')>=F.date_sub(F.current_date(),RETENTION_DAYS)).write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(t)
        print('Applied retention to',t)

schema_change_schema=StructType([StructField('change_id',StringType(),False),StructField('sync_run_id',StringType(),True),StructField('detected_at',TimestampType(),True),StructField('change_type',StringType(),True),StructField('object_level',StringType(),True),StructField('workspace_name',StringType(),True),StructField('lakehouse_name',StringType(),True),StructField('schema_name',StringType(),True),StructField('table_name',StringType(),True),StructField('column_name',StringType(),True),StructField('old_value',StringType(),True),StructField('new_value',StringType(),True),StructField('impact_level',StringType(),True),StructField('review_status',StringType(),True),StructField('review_note',StringType(),True)])
if not spark.catalog.tableExists(f'{CATALOG_SCHEMA}.hist_schema_change'):
    spark.createDataFrame([],schema_change_schema).write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(f'{CATALOG_SCHEMA}.hist_schema_change')
